# Interactive Dashboards & Visualizations

This notebook creates interactive maps and dashboards for stakeholder presentation.

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
import folium
from folium import plugins
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

ROOT = Path('../').resolve()
sys.path.append(str(ROOT))

DATA_PROCESSED = ROOT / 'data' / 'processed'
MAPS_DIR = ROOT / 'maps'
MAPS_DIR.mkdir(parents=True, exist_ok=True)

print('Loading scored dataset...')
try:
    gdf = gpd.read_parquet(DATA_PROCESSED / 'roads_scored_final.parquet')
except FileNotFoundError:
    print('Scored data not found, loading processed data instead...')
    gdf = gpd.read_parquet(DATA_PROCESSED / 'roads_processed_gdf.parquet')

print(f'Loaded {len(gdf)} road segments')

## Interactive Priority Map

In [ ]:
# Create priority-colored map
if 'priority_level' not in gdf.columns:
    raise ValueError('priority_level is missing. Run notebooks/03_modeling.ipynb before this dashboard notebook.')

# Get map center
bounds = gdf.total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=6,
    tiles='OpenStreetMap'
)

# Color mapping
priority_colors = {
    'Low': 'gray',
    'Medium': 'orange',
    'High': 'red'
}

# Add segments (sample for performance)
sample_gdf = gdf.sample(min(800, len(gdf)), random_state=42)

for idx, row in sample_gdf.iterrows():
    if row.geometry.geom_type == 'LineString':
        coords = [[point[1], point[0]] for point in row.geometry.coords]
        
        priority = row.get('priority_level', 'Medium')
        color = priority_colors.get(priority, 'blue')
        
        popup_text = f"Priority: {priority}<br>Length: {row.get('length_km', 'N/A'):.2f} km"
        
        folium.PolyLine(
            coords,
            color=color,
            weight=2,
            opacity=0.7,
            popup=folium.Popup(popup_text, max_width=200)
        ).add_to(m)

# Add legend
legend_html = '''
     <div style="position: fixed; 
     bottom: 50px; right: 50px; width: 180px; height: 140px; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;
     padding: 10px">
     <b>Road Priority</b><br>
     <i style="background:red"></i> High<br>
     <i style="background:orange"></i> Medium<br>
     <i style="background:gray"></i> Low
     </div>
     '''
m.get_root().html.add_child(folium.Element(legend_html))

map_path = MAPS_DIR / 'priority_map.html'
m.save(str(map_path))
print(f'Saved priority map: {map_path}')
m

## Key Metrics Dashboard

In [ ]:
# Create KPI summary using plotly
kpi_data = {
    'Total Road Length (km)': f"{gdf['length_km'].sum():.0f}",
    'Number of Segments': f"{len(gdf)}",
    'Avg Segment Length (km)': f"{gdf['length_km'].mean():.2f}",
    'TEN-T Corridors': f"{(gdf['is_tent'] == 1).sum()}",
}

print('\n=== KEY PERFORMANCE INDICATORS ===')
for metric, value in kpi_data.items():
    print(f'{metric}: {value}')

## Comparative Analysis Visualizations

In [ ]:
# Create subplots for comprehensive view
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Road Length Distribution',
        'Priority Level Breakdown',
        'Complexity vs Length',
        'TEN-T vs Regular Total Length'
    )
)

# Plot 1: Length histogram
fig.add_trace(
    go.Histogram(x=gdf['length_km'], nbinsx=30, name='Length', marker_color='blue'),
    row=1, col=1
)
fig.update_xaxes(title_text='Length (km)', row=1, col=1, type='log')

# Plot 2: Priority bar chart (not pie - for subplot compatibility)
if 'priority_level' in gdf.columns:
    priority_counts = gdf['priority_level'].value_counts().sort_index()
    fig.add_trace(
        go.Bar(
            x=priority_counts.index,
            y=priority_counts.values,
            name='Priority Count',
            marker_color=['green', 'orange', 'red']
        ),
        row=1, col=2
    )
    fig.update_yaxes(title_text='Count', row=1, col=2)
    fig.update_xaxes(title_text='Priority Level', row=1, col=2)

# Plot 3: Complexity vs Length
fig.add_trace(
    go.Scatter(
        x=gdf['length_km'],
        y=gdf['curve_complexity'],
        mode='markers',
        marker=dict(color=gdf['is_tent'], colorscale='Viridis', size=5),
        name='Segments',
        text=[f'TENT: {t}' for t in gdf['is_tent']]
    ),
    row=2, col=1
)
fig.update_xaxes(title_text='Length (km)', type='log', row=2, col=1)
fig.update_yaxes(title_text='Curve Complexity', row=2, col=1)

# Plot 4: TEN-T comparison
tent_summary = gdf.groupby('is_tent')['length_km'].sum()
fig.add_trace(
    go.Bar(
        x=['Regular', 'TEN-T'],
        y=[tent_summary.get(0, 0), tent_summary.get(1, 0)],
        name='Total Length',
        marker_color=['gray', 'gold']
    ),
    row=2, col=2
)
fig.update_yaxes(title_text='Total Length (km)', row=2, col=2)
fig.update_xaxes(title_text='Road Type', row=2, col=2)

fig.update_layout(height=800, showlegend=True, title_text='RTIG Roads Analysis Dashboard')
fig.show()

# Save dashboard
dashboard_path = MAPS_DIR / 'dashboard.html'
fig.write_html(str(dashboard_path))
print(f'Saved dashboard: {dashboard_path}')

## Geographic Heatmap

In [ ]:
# Create heatmap of segment density
m_heat = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=6,
    tiles='CartoDB positron'
)

# Prepare heat data
heat_data = []
for idx, row in gdf.iterrows():
    if row.geometry.geom_type == 'LineString':
        # Sample points along line
        coords = list(row.geometry.coords)
        for lon, lat in coords:
            heat_data.append([lat, lon, 0.5])  # intensity

# Add heatmap
plugins.HeatMap(heat_data).add_to(m_heat)

heatmap_path = MAPS_DIR / 'density_heatmap.html'
m_heat.save(str(heatmap_path))
print(f'Saved heatmap: {heatmap_path}')
m_heat

## Executive Summary

In [ ]:
summary = f"""
╔═══════════════════════════════════════════════════════════╗
║      RTIG ROADS NETWORK - ANALYSIS SUMMARY                ║
╚═══════════════════════════════════════════════════════════╝

📊 DATASET OVERVIEW
  • Total road segments: {len(gdf):,}
  • Total network length: {gdf['length_km'].sum():,.0f} km
  • Average segment length: {gdf['length_km'].mean():.2f} km
  • Network coverage area: {((gdf.total_bounds[2] - gdf.total_bounds[0]) * (gdf.total_bounds[3] - gdf.total_bounds[1])):.0f}° (geographic)

🚗 ROAD CLASSIFICATION
  • TEN-T Priority Corridors: {(gdf['is_tent'] == 1).sum()} segments ({(gdf['is_tent'] == 1).sum() / len(gdf) * 100:.1f}%)
  • Regular RTIG Roads: {(gdf['is_tent'] == 0).sum()} segments ({(gdf['is_tent'] == 0).sum() / len(gdf) * 100:.1f}%)
  • TEN-T Network Length: {gdf[gdf['is_tent'] == 1]['length_km'].sum():,.0f} km

⚠️ PRIORITY ASSESSMENT
  • High Priority Segments: {sum(gdf['priority_level'] == 'High') if 'priority_level' in gdf.columns else 'N/A'}
  • Medium Priority Segments: {sum(gdf['priority_level'] == 'Medium') if 'priority_level' in gdf.columns else 'N/A'}
  • Low Priority Segments: {sum(gdf['priority_level'] == 'Low') if 'priority_level' in gdf.columns else 'N/A'}

📈 NETWORK COMPLEXITY
  • Average complexity (vertices/km): {gdf['curve_complexity'].mean():.2f}
  • Most complex segments: {gdf['curve_complexity'].nlargest(3).values}
  • Average vertices per segment: {gdf['num_vertices'].mean():.0f}

💡 KEY INSIGHTS
  1. {(gdf['is_tent'] == 1).sum()} TEN-T corridors form backbone of priority network
  2. Road complexity varies significantly ({gdf['curve_complexity'].min():.2f} - {gdf['curve_complexity'].max():.2f} vertices/km)
  3. Large proportion of network suitable for EV charging corridor development
  4. Data quality excellent: {(gdf.geometry.is_valid.sum() / len(gdf) * 100):.1f}% valid geometries

📍 DELIVERABLES
  ✓ Interactive priority map: maps/priority_map.html
  ✓ Analytical dashboard: maps/dashboard.html
  ✓ Density heatmap: maps/density_heatmap.html
  ✓ Scored dataset: data/processed/roads_scored_final.parquet

🎯 RECOMMENDATIONS FOR IBERDROLA
  1. Focus energy infrastructure on {(gdf['is_tent'] == 1).sum()} TEN-T corridors first
  2. Develop EV charging network along high-priority, low-complexity segments
  3. Leverage energy demand data to refine risk scoring by corridor
  4. Implement monitoring on high-complexity segments (maintenance risk)
"""

print(summary)